# RunScope 공개 기술 노트: 측면 러닝 영상의 4단계 분석

이 노트북은 RunScope의 데이터 사용 원칙, 4단계 정의, 모델 구조, 평가 방법과 공개 가능한 재현 절차를 한곳에 정리한다. 원본 연구 데이터, 참가자 영상·이미지, 개인 촬영 영상, 파생 landmark/힘판 표, 학습된 가중치는 포함하지 않는다.

> RunScope는 개인 비상업 연구용 프로토타입이며 의료기기나 진단 도구가 아니다. 자세 출력은 부상 위험, 질환 또는 치료 필요성을 판정하지 않는다.

**공개본 갱신일** 2026-11-27 · Riglet 전이 실험과 Twente 착지 감사 추가

## 1. 공개 범위

이 저장소에서 공유하는 것은 소스 코드, 데이터 출처, 참가자 분리 원칙, 모델·평가 설계와 집계 결과뿐이다. 다음 항목은 공개하지 않는다.

- 원본 또는 가공 연구 데이터와 다운로드 압축 파일
- 참가자·사용자 영상, 프레임, 얼굴이나 신체가 보이는 결과 그림
- 참가자별 landmark, 힘판, 모션캡처, 배열 파일
- 학습 가중치, checkpoint, 직렬화된 estimator
- 개인 이메일, 로컬 절대경로, 비밀키, 쿠키, API 토큰

데이터는 각 공식 제공처에서 직접 받아 해당 라이선스·윤리·접근 조건을 따라야 한다. RunScope의 코드 라이선스가 외부 데이터의 재배포 권리를 부여하지 않는다.

In [ ]:
from pathlib import Path
import sys

print('Python:', sys.version.split()[0])
print('Working directory:', Path.cwd().name)
print('This public notebook does not load participant data or model weights.')

## 2. 데이터의 역할

| 데이터 | RunScope에서의 역할 | 영상 모델 직접 학습 | 공개 저장소에 원본 포함 |
|---|---|---:|---:|
| Twente RGB + sensor/force | 측면 영상 포즈와 힘판 접지 정답을 연결한 핵심 개발 데이터 | 예 | 아니오 |
| Reznick continuous locomotion | 관절각 기반 초기 4단계 ML 기준선 | 초기 모델만 | 아니오 |
| Fukuchi RBDS | 고속 러닝 운동학·운동역학 참고 및 향후 합성 포즈 연구 | 아니오 | 아니오 |
| King event-timing 자료 | initial contact와 toe-off 시각 오차 참고 | 아니오 | 아니오 |
| Baker running environments | 환경 차이와 force timing 참고 | 아니오 | 아니오 |
| Riglet 2024 Overground Run | 모션캡처 마커 기반 Landing 학습과 기존 4단계 TCN 전이 실험 | 실험 모델에 사용 | 아니오 |
| 개인 사용자 영상 | 동결 모델의 로컬 추론 확인 | 절대 아니오 | 아니오 |

[Twente 센서 데이터](https://doi.org/10.5281/zenodo.6457662) · [Twente RGB 데이터](https://doi.org/10.5281/zenodo.6644593) · [Fukuchi 러닝 데이터](https://doi.org/10.6084/m9.figshare.4543435) · [Riglet 데이터 논문](https://doi.org/10.1038/s41597-024-03420-y)

Riglet 지상 러닝을 참가자 기준으로 70:15:15 목표에 맞춰 학습 21명, 검증 4명, 테스트 5명으로 나눴다. Riglet으로 미세조정한 TCN은 해당 마커 실험에 보존하고, Twente 영상 기준 모델은 기존 버전을 유지한다.

## 3. 힘판 기준 4단계

한 발의 initial contact(IC)부터 같은 발의 다음 IC까지를 한 보폭으로 본다.

1. **Landing**: 수직 지면반력이 임계값을 넘은 최초 접지부터 짧은 운영 구간
2. **Braking**: Landing 종료부터 전후방 지면반력(AP GRF) 영교차까지
3. **Propulsion**: AP GRF 영교차부터 toe-off까지
4. **Recovery**: toe-off부터 같은 발의 다음 IC까지

Landing은 본래 한 시점인 IC다. 프레임 분류를 위해서만 짧은 구간으로 확장하므로, **IC 이벤트 F1**과 **Landing 구간 F1**은 별도 지표로 보고한다.

In [ ]:
phase_definitions = [
    {'label': 1, 'phase': 'Landing', 'boundary': 'IC to IC + short window'},
    {'label': 2, 'phase': 'Braking', 'boundary': 'Landing end to AP-GRF zero crossing'},
    {'label': 3, 'phase': 'Propulsion', 'boundary': 'AP-GRF zero crossing to toe-off'},
    {'label': 4, 'phase': 'Recovery', 'boundary': 'toe-off to next same-foot IC'},
]
phase_definitions

## 4. 영상 전처리와 입력

1. 고정된 측면 영상을 프레임별로 읽는다.
2. 포즈 추정기로 어깨·골반·무릎·발목과 가능한 경우 heel·toe 계열 landmark를 얻는다.
3. 사람 추적, 좌우 발 일관성, visibility와 비정상 이동량을 검사한다.
4. 좌표를 골반 중심과 몸통 길이로 정규화한다.
5. 좌우 발 궤적에서 보폭 후보를 만들고 한 보폭을 64 sample로 재표본화한다.
6. tracked foot, opposite foot, visibility, trunk 정보를 합쳐 **64×26 시계열**을 만든다.

MediaPipe Pose는 heel과 foot index를 제공한다. 일반 COCO-17 형식의 YOLO Pose는 발목만 제공하므로, 두 포즈 형식을 동일한 발 특징처럼 취급하지 않는다.

## 5. 모델 구성

### Boundary Ridge
보폭 요약 특징으로 IC, propulsion 시작, toe-off/recovery 시작, 다음 IC의 상대 시각을 회귀한다. 표본과 참가자 수가 작아 복잡한 tree 모델보다 참가자 밖 일반화가 안정적인 Ridge가 기준 모델로 선택됐다.

### Temporal TCN
64×26 입력을 dilated residual 1D convolution으로 처리해 각 sample의 4단계 확률을 출력한다. class-weighted loss와 손상 증강을 사용하며, 단조 디코더가 한 보폭 안에서 단계의 생리학적 순서를 강제한다.

### Hybrid
Ridge와 TCN의 단계별 확률을 개발 참가자의 out-of-fold 예측으로 결합한다. 혼합 후에도 단조 순서 제약을 적용한다.

### Landing event model
4단계 구간 분류와 별도로, IC에 가장 가까운 sample을 예측하는 전용 Ridge 모델을 둔다. 이는 힘판이 없는 새 영상에서도 접지 **예측**을 만들 수 있지만, 독립 정답 없이는 그 영상의 F1을 계산할 수 없다.

In [ ]:
def monotonic_phase_order_is_valid(labels):
    """한 보폭에서 허용한 단계 순서의 간단한 공개 예시."""
    order = [4, 1, 2, 3, 4]
    compressed = []
    for label in labels:
        if not compressed or compressed[-1] != label:
            compressed.append(label)
    start = 0 if compressed and compressed[0] == 4 else 1
    return compressed == order[start:start + len(compressed)]

assert monotonic_phase_order_is_valid([4, 4, 1, 1, 2, 3, 3, 4])
assert not monotonic_phase_order_is_valid([1, 3, 2, 4])

## 6. 누출 방지와 분할

프레임이나 보폭을 무작위로 섞으면 같은 사람의 주법이 train과 test에 동시에 들어가 점수가 부풀 수 있다. 따라서 다음 원칙을 지킨다.

- train/validation/test를 **참가자 단위**로 분리한다.
- 모델 선택과 임계값 조정은 개발 참가자 안에서 LOSO 또는 내부 검증으로만 한다.
- 잠금 test 참가자는 최종 확정 전 학습·튜닝에 사용하지 않는다.
- 새 데이터의 8:2 또는 70:15:15 분할도 반드시 참가자 기준으로 수행한다.
- 같은 참가자의 trial을 여러 split에 나누지 않는다.
- 사용자 영상은 추론 뒤 성능이 낮다고 판단해 학습에 편입하지 않는다. 편입하려면 별도 학습 버전을 만들고 기존 test 지위를 폐기한다.

In [ ]:
def participant_disjoint(split):
    groups = [set(split[name]) for name in ('train', 'validation', 'test')]
    return all(groups[i].isdisjoint(groups[j]) for i in range(3) for j in range(i + 1, 3))

synthetic_split = {
    'train': ['participant_A', 'participant_B', 'participant_C'],
    'validation': ['participant_D'],
    'test': ['participant_E'],
}
assert participant_disjoint(synthetic_split)
synthetic_split

## 7. 평가 지표와 현재 해석

| 평가 대상 | 정답 | 보고 지표 |
|---|---|---|
| 힘판 동기 데이터 | force-defined IC와 단계 경계 | Landing event F1, MAE(ms), coverage, 4단계 Macro F1 |
| 사람이 독립 라벨한 영상 | visually annotated IC | visually annotated Landing F1 |
| 정답 없는 사용자 영상 | 없음 | 예측 프레임과 신뢰도만; F1 없음 |

Twente 개발 참가자 기반 Hybrid의 nested meta-LOSO 4단계 Macro F1은 약 **0.725**였다. 별도의 Landing 감사에서는 기존에 예측과 연결된 접지 중 ±30 ms 적중 비율이 **0.632(359/568)**, 전체 자격 기준을 충족한 힘판 접지에 대한 적중 비율이 **0.457(359/786)**, 예측 연결 비율이 **0.723(568/786)**이었다. 세 값은 각각의 분모를 기준으로 한 비율이며, 이벤트 F1은 일대일 대응 결과의 TP·FP·FN으로 계산한다.

이 숫자는 특정 데이터와 프로토콜의 연구 결과이며, 새로운 사람·카메라·속도에서 동일한 성능을 보장하지 않는다. 정답 없는 개인 영상의 실측 F1로 해석해서도 안 된다.

In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else float('nan')

# 집계 숫자로 분모에 따른 차이를 확인하는 예시
audit = {
    'force_events_total': 786,
    'predictions_linked': 568,
    'within_tolerance_total': 359,
}
audit['coverage'] = safe_divide(audit['predictions_linked'], audit['force_events_total'])
audit['conditional_success'] = safe_divide(audit['within_tolerance_total'], audit['predictions_linked'])
audit['end_to_end_success'] = safe_divide(audit['within_tolerance_total'], audit['force_events_total'])
{key: round(value, 3) if isinstance(value, float) else value for key, value in audit.items()}

## 8. 자세 출력: GOOD / IMPROVE / UNKNOWN

4단계 인식과 자세 평가는 별개다. 현재 자세 출력은 학습된 임상 분류기가 아니라 측면 2D 측정의 보수적 스크리닝이다.

- **GOOD(초록)**: 사전에 정한 측면 지표 범위와 신뢰도 조건을 충족
- **IMPROVE(빨강)**: 사전에 정한 범위를 벗어나며 여러 보폭에서 반복
- **UNKNOWN(회색)**: landmark 신뢰도가 낮거나 측면 영상만으로 근거가 부족

후면 관찰이 필요한 무릎 외반, 골반 하강, 뒤꿈치 외번은 측면 영상으로 판정하지 않는다. 자세 전문가 정답이 없으므로 GOOD/IMPROVE의 F1도 현재 계산하지 않는다.

## 9. 공개 코드 실행 안내

이 노트북은 데이터나 가중치를 자동으로 내려받지 않는다. 저장소 루트에서 환경을 만들고 각 명령의 도움말을 먼저 확인한다.

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

python video_v1/analyze_video_end_to_end.py --help
python video_v1/extract_pose_yolo.py --help
python model_v1/predict_phase_csv.py --help
```

일부 스크립트는 사용자가 합법적으로 확보한 데이터와 로컬에서 학습한 artifact를 요구한다. 공개 저장소에는 pretrained RunScope 가중치가 없다. 외부 pose 모델을 받을 때는 해당 제공자의 라이선스와 네트워크 정책을 확인한다.

## 10. 학습과 실험 기록

| 실험 | 결과 | 실험에서 내린 판단 |
|---|---|---|
| Reznick 관절각 기반 초기 4단계 모델 | 내부 참가자 분리 테스트 Macro F1 0.9585 | 관절각 기준선으로 보존 |
| Twente 영상 기반 Ridge·TCN·Hybrid | 개발 Hybrid 4단계 Macro F1 0.725 | 영상 파이프라인의 비교 기준 |
| Twente 보폭 확장 | 중복 제거 후 460 → 1,642보폭. 공통 참가자 LOSO Macro F1 0.706 → 0.701 | 보폭을 늘린 구성에서 개선 근거를 얻지 못해 기존 모델 유지 |
| Riglet 착지 모델 | Histogram Gradient Boosting과 TCN을 검증에서 비교. 선택된 Gradient Boosting은 힘판 확인 구간의 접지 160건에서 F1 1.000, 평균 절대오차 4.0 ms | 평가 범위는 100 Hz 모션캡처와 힘판 확인 구간 |
| Riglet 4단계 전이 | 기존 Twente TCN 가중치에서 Riglet 마커로 미세조정. Riglet 테스트의 연결 보폭 124/160에서 참가자 평균 Macro F1 0.895 | Riglet 마커 조건에서 전이 실험 보존 |

Riglet의 착지 이벤트 실험은 같은 발의 힘판 확인 접지 주변 ±150 ms 구간을 채점했다. 4단계 실험은 연결 보폭의 100 Hz 프레임을 채점했고, 회복의 종료는 데이터 제공자의 마커 기반 다음 접지로 정의했다. 실사용 측면 RGB에서는 별도의 검증이 필요하다.

### Riglet 전이 모델을 Twente에 다시 적용한 결과

| 지표 | 기존 Twente TCN | Riglet 미세조정 TCN |
|---|---:|---:|
| 4단계 프레임 Macro F1 | 0.702 | 0.382 |
| 착지 구간 프레임 F1 | 0.347 | 0.015 |
| 최초 접지 이벤트 F1, 영상 ±1프레임 | 0.471 | 0.038 |
| 최초 접지 시각의 부호 있는 오차 중앙값 | +25 ms | +88 ms |

Twente 역사적 참조의 같은 영상 입력, 보폭과 라벨을 사용했다. 프레임 F1은 연결된 109보폭, 이벤트 F1은 자격 기준을 충족한 전체 힘판 접지 147건 기준이다. 기존 Ridge와 결합 가중치를 유지하고 Hybrid의 TCN만 교체했을 때도 Macro F1은 **0.745 → 0.686**으로 낮아졌다. 사용자 촬영 영상에서는 동결 추론 및 네 단계 출력까지 확인했다. 실측 F1은 독립 주석을 붙인 뒤 계산한다.

## 11. 착지 오차와 시행착오

1. **후보의 의미가 달랐다.** Riglet 학습 343보폭 중 331개에서 후보가 뒤꿈치의 최대 전방 도달과 10 ms 이내였다. 힘판 접지는 후보보다 중앙값 60 ms 늦었다. 원본 힘을 확인한 표본 10보폭에서 후보와 원본 20 N 접지 사이는 중앙값 46.5 ms, 원본 접지에서 후처리 라벨까지는 5.5 ms였다. 전체 간격에서 발을 앞으로 내미는 순간을 후보로 잡은 영향이 컸다.
2. **필터와 샘플링만 바꿔서는 차이가 해소되지 않았다.** Twente 영상 포즈에 6 Hz를 추가한 뒤 대응된 446보폭에서 후보–접지 간격의 변화 중앙값은 0 ms였다. 두 데이터의 평활화 길이를 약 90 ms로 맞춰도 Riglet 후보는 약 60 ms 앞섰다. Twente의 ±1프레임 후보 포함률은 51.5%에서 47.4%로 낮아졌다.
3. **후보가 이미 연결된 보폭으로만 보면 누락이 가려졌다.** Twente 학습군의 전체 힘판 접지 639건을 분모로 삼았을 때 ±1/±3/±5프레임에 같은 발 후보가 있는 비율은 37.1% / 66.0% / 72.8%였다. ±5프레임 밖 174건 중 123건은 두 참가자에게 집중됐다.
4. **좌우 연속성 보정은 전체 성능을 개선하지 못했다.** 영상만으로 보정한 뒤 ±1/±3/±5프레임 포함률은 33.8% / 59.9% / 67.4%였고 오검출 후보가 늘었다. 엄격한 영상 품질 필터는 프레임의 89.4%를 제외했고, 전체 힘판 접지의 88.6%가 제외 프레임에 해당했다. 따라서 필터를 적용해 남은 구간의 점수와 전체 분모 점수를 함께 봤다.
5. **추가 학습 전에 라벨 품질 점검에서 중단했다.** 파일 식별자와 동기화·채점 시각 기준은 일치했고 영상과 모션캡처의 여러 보폭 평균 주기도 가까웠다. 일부 오른발 힘판 접지열에서는 정상 주기의 약 두 배 간격과 한 주기를 둘로 나눈 듯한 간격이 발견됐다. 원본 힘에서 접지 검출의 누락과 중복 여부를 확인하기 전까지 아래의 후속 학습은 보류했다.

## 12. 블라인드 판독과 후속 실험 상태

Twente 학습군에서 7명 × 3속도, 총 21개의 연속 프레임 세트를 준비했다. 판독자는 지정된 발의 신발 밑창이 트레드밀 벨트에 처음 닿아 보이는 프레임을 고른다. 뒤꿈치·앞꿈치 중 먼저 닿은 쪽을 기준으로 하고, 발이 가려져 판단하기 어렵다면 `?`로 남긴다. 판독 뒤 힘판 기준과 프레임·ms 차이, ±1프레임 일치율, 판단 불가율을 계산한다. 선택된 21세트는 작은 판독 표본이므로 이 일치율만으로 모델의 전체 F1이나 0.85 달성 가능성을 확정하지 않는다.

후속 설계는 참가자 분리 LOSO에서 A) 기존 후보, B) 후보 ±5프레임에서 접지를 재선택, C) 모든 프레임에서 접지를 탐색하는 방식이다. B와 C는 같은 뒤꿈치·발끝 좌표, 속도 및 발 주변 RGB 입력을 쓰며, 각 회차의 설정은 해당 학습 참가자 내부에서만 정한다. 성능은 접지 ±1프레임 정밀도·재현율·F1 및 후보 누락으로 보고할 계획이다. **현재 상태는 라벨 감사로 중단이며 A/B/C 학습·평가는 미실행이다.**

새 스마트폰 영상에서는 60/120 fps 측면 원본과 독립 프레임 주석을 함께 확보해 30 fps 입력과 고속 입력을 공정하게 비교할 수 있다.

## 13. 현재 한계

1. 핵심 영상 모델의 독립 참가자 수가 작아 일반화 불확실성이 크다.
2. Landing 오류에는 모델뿐 아니라 동기화, 후보 누락, 좌우 발 교환이 포함된다.
3. 약 30 fps에서는 한 프레임이 약 33 ms라 접지 시각의 양자화 오차가 크다.
4. 보폭 수 증가만으로 새 참가자 일반화 문제가 해결되지는 않는다.
5. 영상 정답과 힘판 정답을 사용한 성능은 각 기준으로 보고한다.
6. 60 fps 이상 고정 측면 촬영은 시각 판독에 도움이 되며, 실측 F1의 계산에는 모델과 독립된 정답이 필요하다.

가장 우선할 개선은 후보 생성 recall, 선택된 후보 주변의 시각 정밀화, 발 landmark 품질, 동기화 감사를 분리해 각각 평가하는 것이다.

## 14. 요약

- RunScope는 측면 러닝 영상에서 pose → stride → 64×26 sequence → 4단계와 Landing event를 예측한다.
- Ridge, TCN, Hybrid를 분리 보존하며 같은 참가자가 학습과 평가에 섞이지 않게 한다.
- 힘판 정답, 사람 라벨 정답, 정답 없는 영상의 결과를 서로 다른 이름으로 보고한다.
- F1은 독립 정답이 있을 때만 계산한다.
- 자세 평가는 GOOD/IMPROVE/UNKNOWN 스크리닝이며 의료 진단이 아니다.
- 공개 저장소는 코드와 문서만 제공하고 원본 데이터·영상·파생 참가자 자료·가중치는 배포하지 않는다.